0. 준비: 라이브러리 설치

In [ ]:
pip install cryptography

1. RSA 공개키/개인키 생성 코드

In [ ]:
from cryptography.hazmat.primitives.asymmetric import rsa
from cryptography.hazmat.primitives import serialization

def generate_keys():
    # 2048비트 RSA 키 생성
    private_key = rsa.generate_private_key(
        public_exponent=65537,
        key_size=2048,
    )

    public_key = private_key.public_key()

    # 개인키를 PEM 형식으로 저장 (비밀번호 없이 예시)
    with open("private_key.pem", "wb") as f:
        f.write(
            private_key.private_bytes(
                encoding=serialization.Encoding.PEM,
                format=serialization.PrivateFormat.PKCS8,
                encryption_algorithm=serialization.NoEncryption(),
            )
        )

    # 공개키를 PEM 형식으로 저장
    with open("public_key.pem", "wb") as f:
        f.write(
            public_key.public_bytes(
                encoding=serialization.Encoding.PEM,
                format=serialization.PublicFormat.SubjectPublicKeyInfo,
            )
        )

    print("키 생성 완료: private_key.pem, public_key.pem")

키 생성 완료: private_key.pem, public_key.pem


2. 공개키 전달

In [4]:
with open("public_key.pem", "r") as f:
    public_key_text = f.read()

public_key_text

'-----BEGIN PUBLIC KEY-----\nMIIBIjANBgkqhkiG9w0BAQEFAAOCAQ8AMIIBCgKCAQEAogS0pkbBLP+vS5CUqGtc\nlGvxj2qw/4IXFJa4GZTHftyCxTR3+qkCI0JRkw+KKDfA9VZr9W2syO4yarbxM/0L\nf+HeGDka8LEdg/A0rvb6RhbqyKSQe9d7ebjbffbxi4YJIlI6qrLvdttsNrsorsTX\nzWlUUzOrp8ffCBZ1lzI5lwalxdxpqQzD48er3w7U/Qfot2WNw3c6ynmBJW/SjhNK\nwQZ1J8dpG33pjKx6ThwfLlR8kjtpeO2XiWjbhxU38sKg0TbYP8NH19okP6afFtiG\nYb+OfI5+Ie7/0MuVwLR2+X0vnsOHbhtnZrgG+46+PgaHi0QnPnwLt1ZoxYVo2FmU\nbQIDAQAB\n-----END PUBLIC KEY-----\n'

3.1. 서버

In [ ]:
import socket
from cryptography.hazmat.primitives import serialization, hashes
from cryptography.hazmat.primitives.asymmetric import padding

# 공개키 로드
with open("public_key.pem", "rb") as f:
    public_key_bytes = f.read()

# 서버 실행
server = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
server.bind(("0.0.0.0", 5000))
server.listen(1)

print("서버 대기중...")

conn, addr = server.accept()
print(f"클라이언트 접속: {addr}")

# 1) 공개키 전송
conn.sendall(public_key_bytes)

# 2) 클라이언트로부터 암호문 받기
encrypted_data = conn.recv(4096)

# 3) 개인키로 복호화
from cryptography.hazmat.primitives import serialization

with open("private_key.pem", "rb") as f:
    private_key = serialization.load_pem_private_key(f.read(), password=None)

plaintext = private_key.decrypt(
    encrypted_data,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None,
    ),
)

print("복호화된 데이터:", plaintext.decode())
conn.close()

3.1. 클라이언트

In [ ]:
SERVER_IP = ""

In [ ]:
import socket
from cryptography.hazmat.primitives import serialization, hashes
from cryptography.hazmat.primitives.asymmetric import padding

client = socket.socket(socket.AF_INET, socket.SOCK_STREAM)
client.connect((SERVER_IP, 5000))

# 1) 서버에서 공개키 받기
public_key_bytes = client.recv(4096)
public_key = serialization.load_pem_public_key(public_key_bytes)

# 2) 암호화할 데이터
plaintext = b"hello secret key"

encrypted = public_key.encrypt(
    plaintext,
    padding.OAEP(
        mgf=padding.MGF1(algorithm=hashes.SHA256()),
        algorithm=hashes.SHA256(),
        label=None,
    ),
)

# 3) 서버로 전송
client.sendall(encrypted)
client.close()
